In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5.5
fig_height = 3.5
fig_format = 'pdf'
fig_dpi = 300
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL2pvbmF0aGFuL0RvY3VtZW50cy9VbkJvc3F1ZS9NYXN0ZXJFQUNEL0N1cnNvcy9NYWNoaW5lTGVhcm5pbmdJL01hY2hpbmVMZWFybmluZ0Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/importlib/_bootstrap.py": 1746004236.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/importlib/_bootstrap_external.py": 1746004236.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/codecs.py": 1746004221.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/encodings/aliases.py": 1746004240.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/encodings/cp437.py": 1746004241.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/encodings/__init__.py": 1746004240.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/encodings/utf_8.py": 1746004242.0, "/Library/Developer/CommandLineT

In [2]:
#| label: fig-logistic-function
#| fig-cap: The logistic, or sigmoid, function.
#| fig-align: center
#| echo: false
#| warning: false

import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-8, 8, 500)
sigma = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 5))

plt.plot(
    z,
    sigma,
    color="#0585d5",
    linewidth=3,
    label=r"$\sigma(z)=\frac{1}{1+e^{-z}}$"
)

plt.axhline(
    0.5,
    color="gray",
    linestyle="--",
    linewidth=1
)

plt.axvline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1
)

plt.scatter(
    0,
    0.5,
    color="red",
    s=60,
    zorder=5
)

plt.annotate(
    r"$\sigma(0)=0.5$",
    xy=(0, 0.5),
    xytext=(1.2, 0.62),
    arrowprops=dict(arrowstyle="->"),
    fontsize=11
)

plt.fill_between(
    z,
    sigma,
    1,
    alpha=0.15,
    color="green",
    label="Positive-class region"
)

plt.fill_between(
    z,
    0,
    sigma,
    alpha=0.15,
    color="orange",
    label="Negative-class region"
)

plt.xlabel(r"Linear score $\mathbf{x}^{T}\boldsymbol{\theta}$")
plt.ylabel(r"Probability $\sigma(\mathbf{x}^{T}\boldsymbol{\theta})$")

plt.xlim(-8, 8)
plt.ylim(0, 1.05)

plt.grid(alpha=0.3)
plt.legend(frameon=False)
plt.tight_layout()

plt.show()

<Figure size 2400x1500 with 1 Axes>

In [3]:
#| label: lst-load-iris

from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

iris = datasets.load_iris(as_frame=True)

list(iris.keys())

['data',
 'target',
 'frame',
 'target_names',
 'DESCR',
 'feature_names',
 'filename',
 'data_module']

In [4]:
#| label: lst-prepare-binary-iris

X = iris.data[["petal width (cm)"]].values
y = iris.target == 2

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [5]:
#| label: lst-train-binary-logistic-regression

log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train, y_train)

LogisticRegression(random_state=42)

In [6]:
#| label: fig-iris-logistic-regression
#| fig-cap: Estimated probabilities and decision boundary for Iris virginica classification.
#| fig-align: center
#| out-width: 100%
#| code-fold: true
#| code-summary: Show code

X_new = np.linspace(0, 3, 1000).reshape(-1, 1)

y_proba = log_reg.predict_proba(X_new)

decision_boundary = X_new[
    y_proba[:, 1] >= 0.5
][0, 0]

plt.figure(figsize=(7, 3.5))

plt.plot(
    X_new,
    y_proba[:, 0],
    "b--",
    linewidth=2,
    label="Not Iris virginica"
)

plt.plot(
    X_new,
    y_proba[:, 1],
    "g-",
    linewidth=2,
    label="Iris virginica"
)

plt.plot(
    [decision_boundary, decision_boundary],
    [0, 1],
    "k:",
    linewidth=2,
    label="Decision boundary"
)

plt.arrow(
    x=decision_boundary,
    y=0.08,
    dx=-0.3,
    dy=0,
    head_width=0.05,
    head_length=0.1,
    fc="b",
    ec="b"
)

plt.arrow(
    x=decision_boundary,
    y=0.92,
    dx=0.3,
    dy=0,
    head_width=0.05,
    head_length=0.1,
    fc="g",
    ec="g"
)

plt.plot(
    X_train[y_train == 0],
    y_train[y_train == 0],
    "bs",
    label="Training: not virginica"
)

plt.plot(
    X_train[y_train == 1],
    y_train[y_train == 1],
    "g^",
    label="Training: virginica"
)

plt.xlabel("Petal width (cm)")
plt.ylabel("Estimated probability")
plt.legend(loc="center left")
plt.axis([0, 3, -0.02, 1.02])
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

<Figure size 2100x1050 with 1 Axes>

In [7]:
#| label: lst-display-decision-boundary

decision_boundary

np.float64(1.6666666666666667)

In [8]:
#| label: lst-logistic-predict

log_reg.predict([[1.7], [1.5]])

array([ True, False])

In [9]:
#| label: lst-logistic-predict-proba

log_reg.predict_proba([[1.7], [1.5]])

array([[0.46703281, 0.53296719],
       [0.65270529, 0.34729471]])

In [10]:
#| label: lst-prepare-two-feature-iris

X = iris.data[
    ["petal length (cm)", "petal width (cm)"]
].values

y = iris.target == 2

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [11]:
#| label: lst-train-two-feature-logistic-regression

log_reg_2d = LogisticRegression(
    C=2,
    random_state=42
)

log_reg_2d.fit(X_train, y_train)

LogisticRegression(C=2, random_state=42)

In [12]:
#| label: fig-logistic-regression-two-features
#| fig-cap: Decision boundary and estimated probabilities using petal length and petal width.
#| fig-align: center
#| out-width: 100%
#| code-fold: true
#| code-summary: Show code

x0, x1 = np.meshgrid(
    np.linspace(2.9, 7, 500),
    np.linspace(0.8, 2.7, 200)
)

X_new = np.c_[x0.ravel(), x1.ravel()]

y_proba = log_reg_2d.predict_proba(X_new)
zz = y_proba[:, 1].reshape(x0.shape)

left_right = np.array([2.9, 7])

boundary = -(
    log_reg_2d.coef_[0, 0] * left_right
    + log_reg_2d.intercept_[0]
) / log_reg_2d.coef_[0, 1]

plt.figure(figsize=(7, 4))

plt.plot(
    X_train[y_train == 0, 0],
    X_train[y_train == 0, 1],
    "bs",
    label="Not Iris virginica"
)

plt.plot(
    X_train[y_train == 1, 0],
    X_train[y_train == 1, 1],
    "g^",
    label="Iris virginica"
)

contour = plt.contour(
    x0,
    x1,
    zz,
    cmap=plt.cm.brg
)

plt.clabel(
    contour,
    inline=True,
    fontsize=8
)

plt.plot(
    left_right,
    boundary,
    "k--",
    linewidth=2,
    label="Decision boundary"
)

plt.text(
    3.5,
    1.27,
    "Not Iris virginica",
    color="navy",
    ha="center"
)

plt.text(
    6.4,
    2.3,
    "Iris virginica",
    color="darkgreen",
    ha="center"
)

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.axis([2.9, 7, 0.8, 2.7])
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

/Users/jonathan/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/jonathan/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/jonathan/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


<Figure size 2100x1200 with 1 Axes>

In [13]:
#| label: lst-prepare-softmax-iris

X = iris.data[
    ["petal length (cm)", "petal width (cm)"]
].values

y = iris.target.values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [14]:
#| label: lst-train-softmax-regression

softmax_reg = LogisticRegression(
    solver="lbfgs",
    C=10,
    random_state=42
)

softmax_reg.fit(X_train, y_train)

LogisticRegression(C=10, random_state=42)

In [15]:
#| label: lst-softmax-parameters

print("Intercepts:")
print(softmax_reg.intercept_)

print("\nCoefficients:")
print(softmax_reg.coef_)

Intercepts:
[ 18.56219292   6.93248241 -25.49467533]

Coefficients:
[[-4.41240489 -2.19019064]
 [-0.15153592 -1.72265407]
 [ 4.5639408   3.9128447 ]]


In [16]:
#| label: lst-softmax-predict-proba

softmax_reg.predict_proba([[5.0, 2.0]])

array([[2.01272527e-06, 8.15683946e-02, 9.18429593e-01]])

In [17]:
#| label: lst-softmax-classes

softmax_reg.classes_

array([0, 1, 2])

In [18]:
#| label: lst-softmax-predict

print(softmax_reg.predict_proba([[5.0, 2.0]]))
prediction = softmax_reg.predict([[5.0, 2.0]])

iris.target_names[prediction]

[[2.01272527e-06 8.15683946e-02 9.18429593e-01]]


array(['virginica'], dtype='<U10')

In [19]:
#| label: fig-softmax-decision-regions
#| fig-cap: Decision regions and estimated probabilities obtained with Softmax Regression.
#| fig-align: center
#| out-width: 100%
#| code-fold: true
#| code-summary: Show code
from matplotlib.colors import ListedColormap

custom_cmap = ListedColormap(
    ["#fafab0", "#9898ff", "#a0faa0"]
)

x0, x1 = np.meshgrid(
    np.linspace(0, 8, 500),
    np.linspace(0, 3.5, 200)
)

X_new = np.c_[x0.ravel(), x1.ravel()]

y_proba = softmax_reg.predict_proba(X_new)
y_predict = softmax_reg.predict(X_new)

zz = y_predict.reshape(x0.shape)
zz1 = y_proba[:, 1].reshape(x0.shape)

plt.figure(figsize=(8,4))

plt.plot(
    X_train[y_train == 2,0],
    X_train[y_train == 2,1],
    "g^",
    label="Iris virginica"
)

plt.plot(
    X_train[y_train == 1,0],
    X_train[y_train == 1,1],
    "bs",
    label="Iris versicolor"
)

plt.plot(
    X_train[y_train == 0,0],
    X_train[y_train == 0,1],
    "yo",
    label="Iris setosa"
)

plt.contourf(
    x0,
    x1,
    zz,
    cmap=custom_cmap,
    alpha=0.35
)

contour = plt.contour(
    x0,
    x1,
    zz1,
    cmap="hot"
)

plt.clabel(
    contour,
    inline=True,
    fontsize=8
)

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")

plt.legend(loc="upper left")

plt.axis([0.5,7,0,3.5])

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

/Users/jonathan/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/jonathan/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/jonathan/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/jonathan/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/jonathan/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/jonathan/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


<Figure size 2400x1200 with 1 Axes>